# Milestone evaluation: base model versus targeted LoRA

Run this notebook yourself in Colab with a **16 GB class or larger NVIDIA GPU**.
A's supplied baseline ran on a Tesla T4. Premium GPU availability varies with
[Colab availability](https://research.google.com/colaboratory/faq.html#gpu-availability).
The user completed all six full runs with this precision revision on a Tesla T4.
All 9,260 full-run prediction rows were verified finite, and paired comparisons
were reproduced locally. See README's measured results. The original VSR item
`vsr:zeroshot:test:426` NaNs were resolved using vision FP32 and language FP16.

Every evaluation subprocess applies this precision change inside `src.eval`,
after base-model/adapter loading. Attention kernels and prompts are unchanged.
This notebook uses a new `evaluation_milestone_vision_fp32` results folder.
Rerun both smoke tests and all six full evaluations; earlier all-FP16 outputs
must not be reused for this comparison.

Put these **two files in the same Google Drive folder**:
- `evaluate.ipynb` — this notebook.
- `evaluation_inputs.zip` — the prepared code, config, evaluation data, and
  milestone adapter bundle. Locally it is built at `data/colab/evaluation_inputs.zip`.

Before starting:
- Set `DRIVE_ROOT` in the first cell to that folder's mounted path. Colab does
  not automatically use the Drive directory containing the open notebook.
- Select a GPU under **Runtime → Change runtime type**.

No GitHub checkout, token, or evaluation-data download is needed. The archive
contains `src/`, `configs/base.yaml`, `requirements.txt`, all three evaluation
datasets and their manifests, the milestone train/validation JSONL files for
separation checks, and the two PEFT adapter files. Packages and the base-model
weights are downloaded when the corresponding cells run.

Execute setup and the two smoke tests first. Inspect them before starting the
six separate full-run cells. Results go directly to a new results subfolder
on Drive; the supplied baseline and training artifacts are left intact.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import importlib.metadata
import json
import shutil
import subprocess
import sys
import time
import tempfile
import zipfile

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/spatial_probe")
BUNDLE = DRIVE_ROOT / "evaluation_inputs.zip"
REPO = Path(tempfile.mkdtemp(prefix="spatial-evaluation-", dir="/content"))
OUT = DRIVE_ROOT / "results/evaluation_milestone_vision_fp32"
assert BUNDLE.is_file(), f"Put evaluation_inputs.zip beside the notebook and set DRIVE_ROOT: {DRIVE_ROOT}"


## Unpack the supplied code and inputs

The bundle is read from Drive and unpacked into fresh runtime storage, avoiding
repeated image reads from Drive. Its fingerprint is saved with the results;
a resumed evaluation must use the same bundle. No files in the archive are
extracted back into the supplied Drive folder.


In [ ]:
with BUNDLE.open("rb") as stream:
    bundle_hash = hashlib.file_digest(stream, "sha256").hexdigest()
fingerprint = OUT / "evaluation_inputs.sha256"
if fingerprint.exists():
    assert fingerprint.read_text().strip() == bundle_hash,             "Bundle changed since previous evaluations; inspect before continuing or choose a new OUT folder"
with zipfile.ZipFile(BUNDLE) as archive:
    for member in archive.infolist():
        path = Path(member.filename)
        assert not path.is_absolute() and ".." not in path.parts, "Unsafe archive path"
    archive.extractall(REPO)
for relative in ("src/eval.py", "src/model_utils.py", "src/stats.py", "configs/base.yaml", "requirements.txt"):
    assert (REPO / relative).is_file(), f"Missing code/config in bundle: {relative}"
OUT.mkdir(parents=True, exist_ok=True)
fingerprint.write_text(bundle_hash + "\n")
print("Bundle SHA-256:", bundle_hash)
print("Runtime files:", REPO)


## Set up the runtime venv

A local `.venv` is created with `--system-site-packages` to reuse Colab's
preinstalled CUDA Torch/torchvision. Only the evaluation packages already
declared in `requirements.txt` are installed into this venv. Torch and
torchvision versions are constrained to the preinstalled versions. Transformers
is pinned to 5.22.1 from A's saved baseline, and PEFT to 0.21.2 from B's adapter
metadata, keeping the adapter's module paths consistent with the supplied run.

This cell also removes Colab's optional `torchao` package, matching B's
successful Colab setup: its preinstalled version conflicted with PEFT.
This affects only this disposable Colab runtime. No extra model dependencies
or quantization are introduced, and no model weights are loaded by this cell.


In [ ]:
gpu = json.loads(subprocess.check_output([sys.executable, "-c", """
import json, torch
assert torch.cuda.is_available(), "Select a GPU runtime before continuing"
p = torch.cuda.get_device_properties(0)
assert p.total_memory >= 14 * 1024**3, "Use a 16 GB class or larger GPU"
print(json.dumps({"name": p.name, "vram_gib": round(p.total_memory / 1024**3, 2),
                  "torch": torch.__version__, "cuda": torch.version.cuda}))
"""], text=True))
print(gpu)
cuda_versions = {name: importlib.metadata.version(name) for name in ("torch", "torchvision")}
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(REPO / ".venv")], check=True)
PYTHON = REPO / ".venv/bin/python"
constraints = REPO / ".venv/colab-cuda-constraints.txt"
constraints.write_text("".join(f"{name}=={version}\n" for name, version in cuda_versions.items())
                       + "transformers==5.22.1\npeft==0.21.2\n")
names = {"transformers", "accelerate", "peft", "numpy", "pillow", "pyyaml"}
requirements = [line.strip() for line in (REPO / "requirements.txt").read_text().splitlines()
                if any(line.lower().startswith(name + ">=") for name in names)]
assert len(requirements) == len(names), "Check evaluation requirements after upstream changes"
subprocess.run([str(PYTHON), "-m", "pip", "install", "-c", str(constraints), *requirements], check=True)

def run(*args):
    print("Running:", " ".join(map(str, args)), flush=True)
    subprocess.run([str(PYTHON), "-u", *map(str, args)], cwd=REPO, check=True)

run("-c", """
import torch, torchvision
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from src import model_utils
assert torch.cuda.is_available(), 'CUDA is unavailable in the venv'
print('Runtime imports passed; no model has been loaded.')
"""
)
OUT.mkdir(parents=True, exist_ok=True)
snapshot = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
(OUT / f"environment_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.txt").write_text(
    f"Code/input bundle SHA-256: {bundle_hash}\nGPU: {json.dumps(gpu)}\n\n" + snapshot)


## Verify the bundled evaluation inputs

All evaluation images, annotations, preparation manifests, and the milestone
adapter are already in the archive. No evaluation-data downloader is run.
This cell verifies the pinned annotation and adapter hashes; the next cell
checks the image bytes, dataset completeness, and train/evaluation separation.

Data and model-cache files on `/content` disappear when the runtime is deleted.
Preparation manifests and results are saved under `OUT` on Drive. After a
runtime reset, rerun setup and preparation, then execute only unfinished
evaluation cells. Existing result files are protected against overwriting.


In [ ]:
DATA = REPO / "data"
ADAPTER = REPO / "runs/lora_targeted/adapter"
adapter_hashes = {
    "adapter_config.json": "c3ba4de249e75e3632c2662b2a9b2f880b7cc7ec0596555d6d77e8d73e85de93",
    "adapter_model.safetensors": "21f6da4280c480f176a64c509dd0bee65060feefcf786e3acdc1f14fa260295f",
}
for name, expected_hash in adapter_hashes.items():
    assert hashlib.sha256((ADAPTER / name).read_bytes()).hexdigest() == expected_hash,             f"{name} differs from the supplied milestone adapter; inspect before evaluating"

frozen = {
    "vsr": ("vsr/zeroshot/test.jsonl", "vsr_zeroshot_test", "914d9156723b912d8f794b179248af865a68c5f81c9d8e3a21d007778cacaad4"),
    "pope": ("pope/coco_pope_random.json", "pope_random", "ac25245170b975a5bdf9080b23fd431dfe6be458bc038259c1f4f09a6bef7994"),
}
for dataset, (annotation, stem, expected_hash) in frozen.items():
    assert hashlib.sha256((DATA / annotation).read_bytes()).hexdigest() == expected_hash,             f"{dataset} annotations differ from the verified milestone selection"
    manifest = DATA / "prepared" / f"{stem}.json"
    destination = OUT / "inputs/prepared" / manifest.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert json.loads(destination.read_text()) == json.loads(manifest.read_text()),                 "Prepared inputs changed since the previous session; inspect before evaluating"
    else:
        shutil.copy2(manifest, destination)


## Check training/evaluation separation

Recheck the supplied milestone train/validation files and selected benchmarks.
Their hashes must match the files audited on the laptop. This checks these
selections only; B's original broader exclusion artifact remains pending.
The fixed 200-question review sample/assignments are also still deferred.


In [ ]:
run("-c", """
import hashlib, json, sys
from pathlib import Path
from src.common import load_config
from src.data._common import read_jsonl
from src.data.vsr import load_vsr
from src.data.pope import load_pope
from src.data.whatsup import load_whatsup, check
from PIL import Image
data, root, adapter = map(Path, sys.argv[1:])
print("What'sUp:", check(load_whatsup(data), data))
for stem, expected_n, expected_images in (('vsr_zeroshot_test', 1222, 715), ('pope_random', 3000, 500)):
    manifest = json.loads((data / 'prepared' / f'{stem}.json').read_text())
    assert manifest['images_verified'] and manifest['n'] == expected_n
    assert len(manifest['images']) == expected_images
    annotation = data / manifest['annotations']['file']
    assert hashlib.sha256(annotation.read_bytes()).hexdigest() == manifest['annotations']['sha256']
    for image in manifest['images']:
        path = data / image['file']
        with path.open('rb') as stream:
            assert hashlib.file_digest(stream, 'sha256').hexdigest() == image['sha256'], f'Image changed: {path}'
        with Image.open(path) as decoded:
            decoded.load()
            assert decoded.size == (image['width'], image['height'])
    print(stem, expected_images, 'images decoded; hashes match')
expected = {
    'train': '42595ac992b7e71423aeedc71f3ad45ab12f599f3197515d8934384959f0626a',
    'val': 'b9c726dba7c40a2fde5522038134b23c225b626243891bcd27738e8a58d66710',
}
ids = {}
for split, digest in expected.items():
    path = root / 'data/coco_spatial' / f'{split}.jsonl'
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, f'{split} data changed'
    ids[split] = {row['coco_image_id'] for _, row in read_jsonl(path)}
assert not ids['train'] & ids['val'], 'Training/validation image overlap'
for name, items in (
    ('vsr_zeroshot_test', load_vsr(data / 'vsr/zeroshot/test.jsonl', data / 'coco')),
    ('pope_random', load_pope(data / 'pope/coco_pope_random.json', data / 'coco')),
):
    selected = {row['coco_image_id'] for row in items}
    for split in ids:
        assert not selected & ids[split], f'{name} overlaps {split} images'
    print(name, len(items), 'questions;', len(selected), 'images; zero train/val overlap')
assert json.loads((adapter / 'adapter_config.json').read_text())['base_model_name_or_path'] == load_config()['model']['id']
print('Adapter base-model identity matches shared config.')
""", DATA, REPO, ADAPTER)

BENCHMARKS = {
    "whatsup": ["--dataset", "whatsup", "--data-root", DATA],
    "vsr_zeroshot_test": ["--dataset", "vsr", "--annotations", DATA / "vsr/zeroshot/test.jsonl", "--image-root", DATA / "coco"],
    "pope_random": ["--dataset", "pope", "--annotations", DATA / "pope/coco_pope_random.json", "--image-root", DATA / "coco"],
}

def evaluate(benchmark, *, adapter=False, limit=None):
    args = ["-m", "src.eval", *BENCHMARKS[benchmark], "--out", OUT]
    if adapter:
        args.extend(["--adapter", ADAPTER])
    if limit is not None:
        args.extend(["--limit", limit])
    run(*args)


## GPU smoke tests — run these before the full evaluations

The first evaluation downloads the approximately 7.5 GB base-model weights.
Each cell runs in its own venv process, releasing GPU memory when it finishes.
Check that both cells finish and save valid results. Eight-question scores are
**not** milestone results. If either cell fails, stop and share its error.


In [ ]:
evaluate("whatsup", limit=8)


In [ ]:
evaluate("whatsup", adapter=True, limit=8)


## Full evaluations — six independent cells

Run these only after both smoke tests succeed. Both baseline and adapter use
FP32 vision and an FP16 language backbone. Shared prompts, pixel limits, attention
kernels, and seed are unchanged. Each successful run
writes a prediction CSV and metric JSON directly to Drive. On reconnect, rerun
only unfinished cells: the evaluator rejects existing outputs by default.
Interruption before a run finishes requires repeating that run from its start.


### whatsup: baseline, then adapter


In [ ]:
evaluate("whatsup")


In [ ]:
evaluate("whatsup", adapter=True)


### vsr_zeroshot_test: baseline, then adapter


In [ ]:
evaluate("vsr_zeroshot_test")


In [ ]:
evaluate("vsr_zeroshot_test", adapter=True)


### pope_random: baseline, then adapter


In [ ]:
evaluate("pope_random")


In [ ]:
evaluate("pope_random", adapter=True)


## Generate the paired before/after table

This requires all six **full** evaluations. It verifies the saved outputs and
matches the questions before computing confidence intervals. The comparison
uses CPU only. Keep the complete output folder together for local inspection.


In [ ]:
baseline = [OUT / "metrics" / f"base_{benchmark}.json" for benchmark in BENCHMARKS]
adapted = [OUT / "metrics" / f"adapted_{benchmark}.json" for benchmark in BENCHMARKS]
run("-m", "src.stats", "--baseline", *baseline, "--adapter", *adapted,
    "--out", OUT / "comparisons")
print((OUT / "comparisons/comparison.md").read_text())
print("Results to retain:", OUT)
